# 1) Data Ingestion Pipeline
## Corrective RAG System — Step 1 of 4

**Goal:** load documents (PDF / DOCX / TXT / CSV) from `data/raw/`, clean the text, and split it into
chunks ready for embedding in the next notebook.

This notebook covers the project requirements:
- Prepare and process documents
- Build an ingestion pipeline: upload, extract, clean, split text


In [1]:
# Run this line once if you have not installed the requirements yet
# %pip install -r requirements.txt -q


In [2]:
import json
import re
from pathlib import Path

import pandas as pd
from pypdf import PdfReader
from docx import Document as DocxDocument

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw documents: {RAW_DIR}")
print(f"Files found  : {[p.name for p in RAW_DIR.glob('*') if p.is_file()]}")


Project root : C:\Users\mahmoud\Desktop\New folder (2)
Raw documents: C:\Users\mahmoud\Desktop\New folder (2)\data\raw
Files found  : ['01_rag_basics.txt', '02_vector_databases.txt', '03_corrective_rag.txt']


## Step 1 — Extract text from each file type

We support `.txt`, `.md`, `.pdf`, `.docx`, `.csv`.

In [3]:
def load_txt(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")


def load_pdf(path: Path) -> str:
    reader = PdfReader(str(path))
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def load_docx(path: Path) -> str:
    doc = DocxDocument(str(path))
    return "\n".join(p.text for p in doc.paragraphs)


def load_csv(path: Path) -> str:
    df = pd.read_csv(path)
    return df.to_string(index=False)


LOADERS = {
    ".txt": load_txt,
    ".md": load_txt,
    ".pdf": load_pdf,
    ".docx": load_docx,
    ".csv": load_csv,
}


def load_document(path: Path) -> str:
    loader = LOADERS.get(path.suffix.lower())
    if loader is None:
        raise ValueError(f"Unsupported file type: {path.suffix}")
    return loader(path)


def load_all_documents(raw_dir: Path) -> list[dict]:
    documents = []
    for path in sorted(raw_dir.glob("*")):
        if path.is_file() and path.suffix.lower() in LOADERS:
            text = load_document(path)
            documents.append({"source": path.name, "text": text})
    return documents


raw_documents = load_all_documents(RAW_DIR)
print(f"Loaded {len(raw_documents)} document(s):")
for doc in raw_documents:
    print(f"  - {doc['source']}: {len(doc['text'])} characters")


Loaded 3 document(s):
  - 01_rag_basics.txt: 2000 characters
  - 02_vector_databases.txt: 2169 characters
  - 03_corrective_rag.txt: 2219 characters


## Step 2 — Clean text

Collapse repeated whitespace/blank lines and strip stray control characters.

In [4]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


for doc in raw_documents:
    doc["text"] = clean_text(doc["text"])

print("Sample after cleaning:\n")
print(raw_documents[0]["text"][:400], "...")


Sample after cleaning:

Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation, or RAG, is a technique that combines a large language model (LLM) with an external
knowledge source. Instead of relying only on the knowledge the model memorized during training, a RAG system
retrieves relevant pieces of text from a document collection at query time and feeds them to the model as context.
This allows the model t ...


## Step 3 — Split into chunks

We use LangChain's `RecursiveCharacterTextSplitter` to split text while preserving context (overlap).

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for doc in raw_documents:
    pieces = text_splitter.split_text(doc["text"])
    for i, piece in enumerate(pieces):
        chunks.append(
            {
                "id": f"{doc['source']}::chunk-{i}",
                "text": piece,
                "source": doc["source"],
                "chunk_index": i,
            }
        )

print(f"Total chunks created: {len(chunks)}")
avg_len = sum(len(c["text"]) for c in chunks) / len(chunks)
print(f"Average chunk length: {avg_len:.0f} characters")


Total chunks created: 11
Average chunk length: 600 characters


## Step 4 — Save processed chunks to disk

We save them to `data/processed/chunks.json` so the next notebook can reuse them without reprocessing.

In [6]:
CHUNKS_PATH = PROCESSED_DIR / "chunks.json"
CHUNKS_PATH.write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved {len(chunks)} chunks to {CHUNKS_PATH}")


Saved 11 chunks to C:\Users\mahmoud\Desktop\New folder (2)\data\processed\chunks.json


## Step 5 — Inspect a sample chunk

In [7]:
import textwrap

sample = chunks[0]
print(f"id     : {sample['id']}")
print(f"source : {sample['source']}")
print("text   :")
print(textwrap.fill(sample["text"], width=100))


id     : 01_rag_basics.txt::chunk-0
source : 01_rag_basics.txt
text   :
Retrieval-Augmented Generation (RAG)  Retrieval-Augmented Generation, or RAG, is a technique that
combines a large language model (LLM) with an external knowledge source. Instead of relying only on
the knowledge the model memorized during training, a RAG system retrieves relevant pieces of text
from a document collection at query time and feeds them to the model as context. This allows the
model to answer questions about private documents, recent events, or domain-specific material that
it was never trained on.
